<a href="https://colab.research.google.com/github/Deelaksha22/LangGraph-Gmail-Integration/blob/main/LnagGraph_Gmail_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U google-api-python-client google-auth-httplib2 google-auth-oauthlib

import os
from google_auth_oauthlib.flow import InstalledAppFlow

os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

SCOPES = [
    "https://www.googleapis.com/auth/gmail.send"
]

flow = InstalledAppFlow.from_client_secrets_file(
    "/content/credentials.json",
    SCOPES
)

flow.redirect_uri = "http://localhost:8080/"

auth_url, _ = flow.authorization_url(
    access_type="offline",
    prompt="consent"
)

print(auth_url)

https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=703999328981-4uk8o89ffrb13vks837qr1pa2mffber3.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send&state=5tzC0i9QDGOaU4cTeNtDJjFmJFWAfD&code_challenge=gqXBleYpOOoW4C-OzrDCpb0Bp3tFbobgOkQdMqeP76U&code_challenge_method=S256&access_type=offline&prompt=consent


In [2]:
redirect_url="http://localhost:8080/?state=5tzC0i9QDGOaU4cTeNtDJjFmJFWAfD&iss=https://accounts.google.com&code=4/0AXlqoi5RtNJB8VvMJhhReUWrYh4hnHTCVrC0WyBNCN37YOR-PeRH0oPPMZnlxWTg1vl3fA&scope=https://www.googleapis.com/auth/gmail.send"

In [5]:
# LangGraph Multi-LLM Failover System with LangSmith Tracing and Gmail Delivery
# GPT-OSS-120B = Primary Generator
# Gemini 3.8 Flash = Primary Tester
# Qwen3-32B = Backup Generator and Tester
# Final verified result is displayed and sent to Gmail

!pip install -q -U langgraph langchain-groq langchain-google-genai langsmith google-api-python-client

import os
import base64
from email.message import EmailMessage
from typing import TypedDict

from google.colab import userdata
from googleapiclient.discovery import build

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END


# ============================================================
# API CONFIGURATION
# ============================================================

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "LangGraph-Multi-LLM"


# ============================================================
# LLM CONFIGURATION
# ============================================================

llm1 = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

llm2 = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

llm3 = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0
)


# ============================================================
# OAUTH CREDENTIALS RETRIEVAL
# ============================================================

# We complete the authorization flow using the flow from cell 1 and the redirect_url from cell 2
flow.fetch_token(authorization_response=redirect_url)
creds = flow.credentials


# ============================================================
# GMAIL SERVICE
# ============================================================

gmail_service = build(
    "gmail",
    "v1",
    credentials=creds,
    cache_discovery=False
)


# ============================================================
# EMAIL FUNCTION
# ============================================================

def send_email(to_email, subject, body):

    message = EmailMessage()

    message["To"] = to_email
    message["Subject"] = subject

    message.set_content(body)

    encoded_message = base64.urlsafe_b64encode(
        message.as_bytes()
    ).decode()

    response = gmail_service.users().messages().send(
        userId="me",
        body={"raw": encoded_message}
    ).execute()

    return response


# ============================================================
# LANGGRAPH STATE
# ============================================================

class State(TypedDict):
    question: str
    answer: str
    test_result: str
    generator: str
    tester: str
    failed_llms: list[str]
    generator_failed: bool
    tester_failed: bool


# ============================================================
# RESPONSE TEXT HANDLER
# ============================================================

def get_text(response):

    content = response.content

    if isinstance(content, str):
        return content

    if isinstance(content, list):

        text = ""

        for item in content:

            if isinstance(item, str):
                text += item

            elif isinstance(item, dict) and "text" in item:
                text += item["text"]

        return text

    return str(content)


# ============================================================
# LLM 1 - PRIMARY GENERATOR
# ============================================================

def llm1_generator(state: State):

    try:

        response = llm1.invoke(
            f"""You are the primary answer generator.

Answer the following question accurately and completely.

Question:
{state["question"]}

Return only the answer."""
        )

        answer = get_text(response).strip()

        if not answer:
            raise Exception("Empty response")

        return {
            "answer": answer,
            "generator": "LLM 1 - GPT-OSS-120B",
            "generator_failed": False
        }

    except Exception:

        return {
            "answer": "",
            "generator": "",
            "generator_failed": True,
            "failed_llms": state["failed_llms"] + [
                "LLM 1 - GPT-OSS-120B"
            ]
        }


# ============================================================
# GENERATOR ROUTER
# ============================================================

def generator_route(state: State):

    if state["generator_failed"]:
        return "llm3_generator"

    return "llm2_tester"


# ============================================================
# LLM 3 - BACKUP GENERATOR
# ============================================================

def llm3_generator(state: State):

    try:

        response = llm3.invoke(
            f"""You are the backup answer generator.

The primary generator failed.

Generate the best possible answer to this question.

Question:
{state["question"]}

Return only the answer."""
        )

        answer = get_text(response).strip()

        if not answer:
            raise Exception("Empty response")

        return {
            "answer": answer,
            "generator": "LLM 3 - Qwen3-32B",
            "generator_failed": False
        }

    except Exception:

        return {
            "answer": "",
            "generator": "LLM 3 - Qwen3-32B",
            "generator_failed": True,
            "failed_llms": state["failed_llms"] + [
                "LLM 3 - Qwen3-32B"
            ]
        }


# ============================================================
# LLM 2 - PRIMARY TESTER
# ============================================================

def llm2_tester(state: State):

    try:

        response = llm2.invoke(
            f"""You are the answer verification AI.

Question:
{state["question"]}

Generated Answer:
{state["answer"]}

Check the answer for:

- Factual correctness
- Completeness
- Logical correctness
- Relevance
- Incorrect information

Return:

RESULT: PASS or FAIL
REASON: explanation"""
        )

        result = get_text(response).strip()

        if not result:
            raise Exception("Empty response")

        return {
            "test_result": result,
            "tester": "LLM 2 - Gemini 3.8 Flash",
            "tester_failed": False
        }

    except Exception:

        return {
            "test_result": "",
            "tester": "",
            "tester_failed": True,
            "failed_llms": state["failed_llms"] + [
                "LLM 2 - Gemini 3.8 Flash"
            ]
        }


# ============================================================
# TESTER ROUTER
# ============================================================

def tester_route(state: State):

    if state["tester_failed"]:
        return "llm3_tester"

    return "end"


# ============================================================
# LLM 3 - BACKUP TESTER
# ============================================================

def llm3_tester(state: State):

    try:

        response = llm3.invoke(
            f"""You are the backup answer verification AI.

Question:
{state["question"]}

Generated Answer:
{state["answer"]}

Check:

- Accuracy
- Completeness
- Logic
- Relevance
- Incorrect information

Return:

RESULT: PASS or FAIL
REASON: explanation"""
        )

        result = get_text(response).strip()

        if not result:
            raise Exception("Empty response")

        return {
            "test_result": result,
            "tester": "LLM 3 - Qwen3-32B"
        }

    except Exception:

        return {
            "test_result": "Unable to verify answer.",
            "tester": "LLM 3 - Qwen3-32B",
            "failed_llms": state["failed_llms"] + [
                "LLM 3 - Qwen3-32B"
            ]
        }


# ============================================================
# BUILD LANGGRAPH
# ============================================================

graph = StateGraph(State)

graph.add_node(
    "llm1_generator",
    llm1_generator
)

graph.add_node(
    "llm3_generator",
    llm3_generator
)

graph.add_node(
    "llm2_tester",
    llm2_tester
)

graph.add_node(
    "llm3_tester",
    llm3_tester
)


graph.add_edge(
    START,
    "llm1_generator"
)


graph.add_conditional_edges(
    "llm1_generator",
    generator_route,
    {
        "llm2_tester": "llm2_tester",
        "llm3_generator": "llm3_generator"
    }
)


graph.add_edge(
    "llm3_generator",
    "llm2_tester"
)


graph.add_conditional_edges(
    "llm2_tester",
    tester_route,
    {
        "llm3_tester": "llm3_tester",
        "end": END
    }
)


graph.add_edge(
    "llm3_tester",
    END
)


app = graph.compile()


# ============================================================
# USER QUESTION
# ============================================================

question = input(
    "Enter your question: "
)


# ============================================================
# RUN LANGGRAPH
# ============================================================

result = app.invoke({

    "question": question,

    "answer": "",

    "test_result": "",

    "generator": "",

    "tester": "",

    "failed_llms": [],

    "generator_failed": False,

    "tester_failed": False
})


# ============================================================
# DISPLAY RESULT
# ============================================================

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)

print(result["answer"])

print("\nGenerator:")
print(result["generator"])

print("\nTester:")
print(result["tester"])


if result["failed_llms"]:

    print(
        "\nFailed LLM(s):",
        ", ".join(result["failed_llms"])
    )

else:

    print("\nFailed LLM(s): None")


print("\nTest Result:")
print(result["test_result"])


# ============================================================
# SEND RESULT TO EMAIL
# ============================================================

EMAIL_TO = "deelaksha63@gmail.com"

email_body = f"""
LangGraph Multi-LLM Result

==================================================
QUESTION
==================================================

{question}


==================================================
FINAL ANSWER
==================================================

{result["answer"]}


==================================================
GENERATOR
==================================================

{result["generator"]}


==================================================
TESTER
==================================================

{result["tester"]}


==================================================
FAILED LLM(S)
==================================================

{", ".join(result["failed_llms"]) if result["failed_llms"] else "None"}


==================================================
TEST RESULT
==================================================

{result["test_result"]}


==================================================
LANGSMITH PROJECT
==================================================

LangGraph-Multi-LLM
"""


email_response = send_email(
    EMAIL_TO,
    "LangGraph Multi-LLM Answer",
    email_body
)


print("\n" + "=" * 60)
print("EMAIL")
print("=" * 60)

print("Email sent successfully!")
print("Message ID:", email_response["id"])

Enter your question: who is ai engineer

FINAL ANSWER
An AI engineer is a technology professional who designs, develops, and deploys artificial‑intelligence solutions. They apply machine‑learning, deep‑learning, and data‑science techniques to build models, create algorithms, and integrate AI capabilities into software applications, products, or services. Their work typically involves data preprocessing, model training and evaluation, software engineering, system architecture, and often the deployment and monitoring of AI systems in production environments.

Generator:
LLM 1 - GPT-OSS-120B

Tester:
LLM 2 - Gemini 3.8 Flash

Failed LLM(s): None

Test Result:
RESULT: PASS
REASON: The generated answer provides an accurate, complete, logically structured, and relevant definition of an AI engineer, correctly detailing their core responsibilities, skill sets, and role within the technology sector without any factual errors.

EMAIL
Email sent successfully!
Message ID: 1a11b0eb21fae18d
